In [1]:
from pathlib import Path

from html import escape

import json

import subprocess

from PIL import Image





ROOT = Path(".")

MEDIA_SITE = ROOT / "media-site"

ANIM_DIR = MEDIA_SITE / "animations"

OUT = MEDIA_SITE / "index.html"

CDN_BASE = "https://media-math.pages.dev"



def media_url(path: Path) -> str:
    rel = path.relative_to(MEDIA_SITE).as_posix()
    return f"{CDN_BASE.rstrip('/')}/{rel}"



VIDEO_EXTS = {".webm", ".mp4"}

IMAGE_EXTS = {".gif"}





def get_image_size(path: Path) -> tuple[int, int] | None:

    try:

        with Image.open(path) as img:

            return img.size

    except Exception:

        return None





def get_video_size(path: Path) -> tuple[int, int] | None:

    try:

        cmd = [

            "ffprobe",

            "-v", "error",

            "-select_streams", "v:0",

            "-show_entries", "stream=width,height",

            "-of", "json",

            str(path),

        ]



        result = subprocess.run(

            cmd,

            check=True,

            capture_output=True,

            text=True,

        )



        data = json.loads(result.stdout)

        stream = data["streams"][0]



        return int(stream["width"]), int(stream["height"])



    except Exception:

        return None





def get_media_size(path: Path) -> tuple[int, int] | None:

    ext = path.suffix.lower()



    if ext in IMAGE_EXTS:

        return get_image_size(path)



    if ext in VIDEO_EXTS:

        return get_video_size(path)



    return None





def classify_media(path: Path) -> str:

    size = get_media_size(path)



    if size is None:

        return "unknown"



    w, h = size



    if w <= 0 or h <= 0:

        return "unknown"



    ratio = w / h



    if 0.92 <= ratio <= 1.08:

        return "square"



    if ratio > 1.08:

        return "horizontal"



    return "vertical"





def make_media_html(path: Path, title: str) -> str:

    url = media_url(path)

    ext = path.suffix.lower()



    if ext == ".gif":

        return f'<img src="{escape(url)}" alt="{escape(title)}">'



    video_type = "video/webm" if ext == ".webm" else "video/mp4"



    return f"""

        <video autoplay muted loop playsinline preload="metadata">

          <source src="{escape(url)}" type="{video_type}">

        </video>

    """





def make_card(path: Path) -> str:

    title = path.relative_to(ANIM_DIR).as_posix()

    size = get_media_size(path)



    if size is None:

        size_label = "unknown size"

    else:

        size_label = f"{size[0]}×{size[1]}"



    media = make_media_html(path, title)



    return f"""

      <article class="card">

        <h2 class="card-title">{escape(title)}</h2>

        <div class="card-meta">{escape(size_label)}</div>

        {media}

      </article>

    """





def make_section(title: str, files: list[Path], css_class: str) -> str:

    if not files:

        return ""



    cards = "\n".join(make_card(path) for path in files)



    return f"""

    <section class="gallery-section gallery-section--{css_class}">

      <div class="section-head">

        <h2>{escape(title)}</h2>

        <span>{len(files)} file(s)</span>

      </div>



      <div class="grid grid--{css_class}">

        {cards}

      </div>

    </section>

    """





files = sorted(

    list(ANIM_DIR.rglob("*.gif")) +

    list(ANIM_DIR.rglob("*.webm")) +

    list(ANIM_DIR.rglob("*.mp4"))

)



groups = {

    "square": [],

    "horizontal": [],

    "vertical": [],

    "unknown": [],

}



for path in files:

    groups[classify_media(path)].append(path)





html = f"""<!doctype html>

<html lang="ru">

<head>

  <meta charset="utf-8">

  <meta name="viewport" content="width=device-width, initial-scale=1">

  <title>HUD Animation Gallery</title>



  <style>

    :root {{

      --bg: #02070d;

      --panel: rgba(5, 18, 28, 0.78);

      --cyan: #5af0ff;

      --cyan-soft: rgba(90, 240, 255, 0.28);

      --text: #d8fbff;

      --muted: #77b8c4;

    }}



    * {{

      box-sizing: border-box;

    }}



    body {{

      margin: 0;

      min-height: 100vh;

      background:

        radial-gradient(circle at 20% 0%, rgba(90, 240, 255, 0.12), transparent 34%),

        radial-gradient(circle at 80% 20%, rgba(90, 255, 170, 0.08), transparent 30%),

        var(--bg);

      color: var(--text);

      font-family: Menlo, Consolas, monospace;

    }}



    .page {{

      width: min(1500px, calc(100vw - 32px));

      margin: 0 auto;

      padding: 28px 0 48px;

    }}



    header {{

      margin-bottom: 28px;

      border-bottom: 1px solid var(--cyan-soft);

      padding-bottom: 16px;

    }}



    h1 {{

      margin: 0 0 8px;

      font-size: 22px;

      letter-spacing: 0.08em;

      color: var(--cyan);

    }}



    .subtitle {{

      color: var(--muted);

      font-size: 13px;

    }}



    .gallery-section {{

      margin: 0 0 42px;

    }}



    .section-head {{

      display: flex;

      align-items: baseline;

      justify-content: space-between;

      gap: 16px;

      margin: 0 0 14px;

      padding-bottom: 8px;

      border-bottom: 1px solid rgba(90, 240, 255, 0.18);

    }}



    .section-head h2 {{

      margin: 0;

      font-size: 15px;

      color: var(--cyan);

      letter-spacing: 0.12em;

      text-transform: uppercase;

    }}



    .section-head span {{

      color: var(--muted);

      font-size: 12px;

      white-space: nowrap;

    }}



    .grid {{

      display: grid;

      gap: 18px;

      align-items: start;

    }}



    .grid--square {{

      grid-template-columns: repeat(auto-fill, minmax(220px, 1fr));

    }}



    .grid--horizontal {{

      grid-template-columns: repeat(auto-fill, minmax(420px, 1fr));

    }}



    .grid--vertical {{

      grid-template-columns: repeat(auto-fill, minmax(180px, 260px));

    }}



    .grid--unknown {{

      grid-template-columns: repeat(auto-fill, minmax(320px, 1fr));

    }}



    .card {{

      background: var(--panel);

      border: 1px solid var(--cyan-soft);

      box-shadow: 0 0 26px rgba(90, 240, 255, 0.08);

      padding: 10px;

    }}



    .card-title {{

      font-size: 12px;

      color: var(--cyan);

      margin: 0 0 4px;

      letter-spacing: 0.04em;

      white-space: nowrap;

      overflow: hidden;

      text-overflow: ellipsis;

    }}



    .card-meta {{

      margin-bottom: 8px;

      font-size: 11px;

      color: var(--muted);

    }}



    .card img,

    .card video {{

      display: block;

      width: 100%;

      height: auto;

      background: #000;

      border: 1px solid rgba(90, 240, 255, 0.18);

      cursor: pointer;

    }}



    .fullscreen-viewer {{

      position: fixed;

      inset: 0;

      z-index: 10000;

      display: flex;

      flex-direction: column;

      align-items: center;

      justify-content: center;

      gap: 12px;

      padding: 24px;

      background: rgba(2, 7, 13, 0.96);

    }}



    .fullscreen-viewer[hidden] {{

      display: none;

    }}



    .fullscreen-close {{

      position: absolute;

      top: 16px;

      right: 20px;

      width: 40px;

      height: 40px;

      border: 1px solid var(--cyan-soft);

      border-radius: 4px;

      background: rgba(5, 18, 28, 0.9);

      color: var(--cyan);

      font: 24px/1 Menlo, Consolas, monospace;

      cursor: pointer;

    }}



    .fullscreen-close:hover {{

      background: rgba(90, 240, 255, 0.12);

    }}



    .fullscreen-caption {{

      max-width: min(96vw, 1200px);

      font-size: 12px;

      color: var(--muted);

      text-align: center;

      letter-spacing: 0.04em;

      word-break: break-all;

    }}



    .fullscreen-stage {{

      display: flex;

      align-items: center;

      justify-content: center;

      width: 100%;

      flex: 1;

      min-height: 0;

    }}



    .fullscreen-stage img,

    .fullscreen-stage video {{

      display: block;

      max-width: min(96vw, 100%);

      max-height: min(86vh, 100%);

      width: auto;

      height: auto;

      object-fit: contain;

      background: #000;

      border: 1px solid rgba(90, 240, 255, 0.28);

      box-shadow: 0 0 40px rgba(90, 240, 255, 0.15);

    }}



    @media (max-width: 760px) {{

      .grid--horizontal,

      .grid--vertical,

      .grid--square,

      .grid--unknown {{

        grid-template-columns: 1fr;

      }}



      .page {{

        width: min(100vw - 18px, 100%);

        padding-top: 18px;

      }}

    }}

  </style>

</head>



<body>

  <main class="page">

    <header>

      <h1>HUD ANIMATION GALLERY</h1>

      <div class="subtitle">{len(files)} animation files · {escape(CDN_BASE)}/animations</div>

    </header>



    {make_section("Square animations", groups["square"], "square")}

    {make_section("Horizontal animations", groups["horizontal"], "horizontal")}

    {make_section("Vertical animations", groups["vertical"], "vertical")}

    {make_section("Unknown / unclassified", groups["unknown"], "unknown")}

  </main>



  <div id="fullscreenViewer" class="fullscreen-viewer" hidden aria-hidden="true">

    <button type="button" class="fullscreen-close" aria-label="Закрыть">×</button>

    <div class="fullscreen-caption"></div>

    <div class="fullscreen-stage"></div>

  </div>



  <script>

    (function () {{

      const viewer = document.getElementById("fullscreenViewer");

      const stage = viewer.querySelector(".fullscreen-stage");

      const caption = viewer.querySelector(".fullscreen-caption");

      const closeBtn = viewer.querySelector(".fullscreen-close");



      function mediaSrc(media) {{

        if (media.tagName === "VIDEO") {{

          const source = media.querySelector("source");

          return source?.src || media.currentSrc || "";

        }}

        return media.currentSrc || media.src || "";

      }}



      function openFullscreen(media) {{

        const card = media.closest(".card");

        const title = card?.querySelector(".card-title")?.textContent?.trim() || "";

        const src = mediaSrc(media);

        if (!src) return;



        stage.replaceChildren();

        let el;

        if (media.tagName === "VIDEO") {{

          el = document.createElement("video");

          el.src = src;

          el.autoplay = true;

          el.muted = true;

          el.loop = true;

          el.playsInline = true;

          el.controls = true;

        }} else {{

          el = document.createElement("img");

          el.src = src;

          el.alt = media.alt || title;

        }}

        stage.appendChild(el);

        caption.textContent = title;

        viewer.hidden = false;

        viewer.setAttribute("aria-hidden", "false");

        document.body.style.overflow = "hidden";

        if (el.tagName === "VIDEO") el.play().catch(() => {{}});

      }}



      function closeFullscreen() {{

        const video = stage.querySelector("video");

        if (video) {{

          video.pause();

        }}

        stage.replaceChildren();

        viewer.hidden = true;

        viewer.setAttribute("aria-hidden", "true");

        document.body.style.overflow = "";

      }}



      document.querySelectorAll(".card img, .card video").forEach(media => {{

        media.addEventListener("click", evt => {{

          evt.preventDefault();

          evt.stopPropagation();

          openFullscreen(media);

        }});

      }});



      closeBtn.addEventListener("click", closeFullscreen);

      viewer.addEventListener("click", evt => {{

        if (evt.target === viewer || evt.target === stage) closeFullscreen();

      }});

      document.addEventListener("keydown", evt => {{

        if (evt.key === "Escape" && !viewer.hidden) closeFullscreen();

      }});

    }})();

  </script>

</body>

</html>

"""



OUT.write_text(html, encoding="utf-8")



print(f"Saved: {OUT}")

print(f"Found files: {len(files)}")

print(f"Square: {len(groups['square'])}")

print(f"Horizontal: {len(groups['horizontal'])}")

print(f"Vertical: {len(groups['vertical'])}")

print(f"Unknown: {len(groups['unknown'])}")



for group_name, group_files in groups.items():

    print()

    print(group_name.upper())

    for f in group_files:

        size = get_media_size(f)

        size_label = f"{size[0]}x{size[1]}" if size else "unknown"

        print(f"  {f} [{size_label}]")


Saved: media-site/index.html
Found files: 254
Square: 93
Horizontal: 152
Vertical: 9
Unknown: 0

SQUARE
  media-site/animations/HUD/Radar_Sweep/radar_sweep_v1.webm [1200x1200]
  media-site/animations/Math/aizawa_attractor_tube_v1.webm [1200x1200]
  media-site/animations/Math/astroidal_ellipsoid_v1_grid.webm [1200x1200]
  media-site/animations/Math/atomic_orbital_lobes_v1_grid.webm [1200x1200]
  media-site/animations/Math/borromean_rings_v1.webm [1200x1200]
  media-site/animations/Math/borromean_rings_v1_grid.webm [1200x1200]
  media-site/animations/Math/bour_surface_v1_grid.webm [1200x1200]
  media-site/animations/Math/boy_surface_v1.webm [1200x1200]
  media-site/animations/Math/boy_surface_v2_grid.webm [1200x1200]
  media-site/animations/Math/breather_surface_v1_grid.webm [1200x1200]
  media-site/animations/Math/catalan_surface_v1_grid.webm [1200x1200]
  media-site/animations/Math/catenoid_v1_grid.webm [1200x1200]
  media-site/animations/Math/cinquefoil_knot_tube_v1.webm [1200x1200]
 